# 03 — Quantitative Extraction & Validation

**Phase 3 deliverable.**

Validates the rule-based metric extractor against CSV ground truth.
Reports exact match rate, MAPE, coverage, and a failure taxonomy.

## Goals
- Run extraction on all 50 companies × 3 years
- Compare to CSV ground truth using `src/extraction/validator.py`
- Achieve >90% exact match rate
- Classify failures: table_not_found, wrong_row, value_parsing_error, wrong_column, section_misidentified
- Confirm SQLite DB is populated correctly

In [1]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src import config
from src.db.schema import get_connection

# Connect to the database
conn = get_connection(config.DB_PATH)
print(f"Database: {config.DB_PATH}")
print(f"Size: {config.DB_PATH.stat().st_size / 1024:.0f} KB")


Database: /home/conderafael/Programing/Portfolio/cvm-intelligence/data/cvm_metrics.db
Size: 277476 KB


## 1. Overall Statistics

In [2]:
# High-level counts
filings = pd.read_sql("SELECT * FROM filings", conn)
metrics = pd.read_sql("SELECT * FROM metrics", conn)
companies = pd.read_sql("SELECT * FROM companies", conn)

print(f"Companies:  {len(companies)}")
print(f"Filings:    {len(filings)}")
print(f"  ITR:      {(filings.filing_type=='ITR').sum()}")
print(f"  DFP:      {(filings.filing_type=='DFP').sum()}")
print(f"")
print(f"Extraction status:")
print(filings.extraction_status.value_counts().to_string())
print(f"")
print(f"Metric records: {len(metrics)}")
print(f"Match status breakdown:")
print(metrics.match_status.value_counts().to_string())


Companies:  49
Filings:    686
  ITR:      504
  DFP:      182

Extraction status:
extraction_status
success    632
partial     33
failed      21

Metric records: 4651
Match status breakdown:
match_status
exact       4425
mismatch     141
missing       70
close         15


In [3]:
# Overall exact+close match rate and coverage
validated = metrics[metrics.match_status != 'missing']
total_records = len(metrics)
exact_close = metrics[metrics.match_status.isin(['exact','close'])].shape[0]
found = len(validated)

match_rate = exact_close / total_records * 100
coverage = found / total_records * 100
mape = validated[validated.percentage_error.notna()]['percentage_error'].mean() * 100

print(f"Exact+close match rate:  {match_rate:.1f}%  (target: >90%)")
print(f"Coverage (non-missing):  {coverage:.1f}%")
print(f"MAPE (where found):      {mape:.4f}%")
print(f"")
print(f"Total metric slots:  {total_records}")
print(f"  → exact:           {(metrics.match_status=='exact').sum()}")
print(f"  → close (<1% err): {(metrics.match_status=='close').sum()}")
print(f"  → mismatch:        {(metrics.match_status=='mismatch').sum()}")
print(f"  → missing:         {(metrics.match_status=='missing').sum()}")


Exact+close match rate:  95.5%  (target: >90%)
Coverage (non-missing):  98.5%
MAPE (where found):      6.4351%

Total metric slots:  4651
  → exact:           4425
  → close (<1% err): 15
  → mismatch:        141
  → missing:         70


## 2. Per-Metric Accuracy

In [4]:
metric_stats = (
    metrics.groupby('metric_name')
    .apply(lambda g: pd.Series({
        'total':        len(g),
        'exact':        (g.match_status == 'exact').sum(),
        'close':        (g.match_status == 'close').sum(),
        'mismatch':     (g.match_status == 'mismatch').sum(),
        'missing':      (g.match_status == 'missing').sum(),
        'match_rate_%': (g.match_status.isin(['exact','close'])).sum() / len(g) * 100,
        'avg_pct_err':  g[g.percentage_error.notna()]['percentage_error'].mean() * 100,
    }))
    .reset_index()
    .sort_values('match_rate_%', ascending=False)
)

print(metric_stats.to_string(index=False))


        metric_name  total  exact  close  mismatch  missing  match_rate_%  avg_pct_err
       total_equity  666.0  646.0    6.0      13.0      1.0     97.897898     0.656890
       total_assets  666.0  647.0    0.0      15.0      4.0     97.147147     0.478668
            revenue  666.0  638.0    8.0      13.0      7.0     96.996997     4.037264
       gross_profit  666.0  637.0    0.0      22.0      7.0     95.645646     4.453629
operating_cash_flow  655.0  625.0    0.0      13.0     17.0     95.419847    19.305134
               cogs  666.0  618.0    1.0      15.0     32.0     92.942943     6.408016
         net_income  666.0  614.0    0.0      50.0      2.0     92.192192    10.166400


In [5]:
fig = px.bar(
    metric_stats,
    x='metric_name', y='match_rate_%',
    color='match_rate_%',
    color_continuous_scale='RdYlGn',
    range_color=[0, 100],
    title='Match Rate by Metric (Exact + Close, %)',
    labels={'metric_name': 'Metric', 'match_rate_%': 'Match Rate (%)'},
)
fig.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='90% target')
fig.update_layout(showlegend=False)
fig.show()


## 3. Per-Company Match Rates

In [6]:
company_metrics = pd.read_sql("""
    SELECT f.ticker, m.metric_name, m.match_status, m.percentage_error
    FROM metrics m
    JOIN filings f ON m.filing_id = f.filing_id
""", conn)

company_stats = (
    company_metrics.groupby('ticker')
    .apply(lambda g: pd.Series({
        'filings':      g.filing_id.nunique() if 'filing_id' in g else 0,
        'total':        len(g),
        'match_rate_%': g.match_status.isin(['exact','close']).sum() / len(g) * 100,
        'missing_%':    (g.match_status == 'missing').sum() / len(g) * 100,
    }))
    .reset_index()
    .sort_values('match_rate_%', ascending=False)
)

# Merge in sector
company_stats = company_stats.merge(companies[['ticker','sector']], on='ticker', how='left')
print(f"Companies with >90% match: {(company_stats['match_rate_%'] >= 90).sum()}/{len(company_stats)}")
print()
print(company_stats[['ticker','sector','total','match_rate_%','missing_%']].to_string(index=False))


Companies with >90% match: 41/49

ticker               sector  total  match_rate_%  missing_%
 ABEV3              Bebidas   91.0    100.000000   0.000000
 ALPA4             Calçados  119.0    100.000000   0.000000
 B3SA3 Serviços Financeiros   91.0    100.000000   0.000000
 BRFS3            Alimentos   84.0    100.000000   0.000000
 BEEF3            Alimentos   84.0    100.000000   0.000000
 CMIG4     Energia Elétrica   84.0    100.000000   0.000000
 ELET3     Energia Elétrica   26.0    100.000000   0.000000
 CSNA3           Siderurgia  112.0    100.000000   0.000000
 CMIN3            Mineração  105.0    100.000000   0.000000
 EMBR3          Aeronáutico   91.0    100.000000   0.000000
 ENEV3     Energia Elétrica  105.0    100.000000   0.000000
 EGIE3     Energia Elétrica   98.0    100.000000   0.000000
 JBSS3            Alimentos   91.0    100.000000   0.000000
 HYPE3         Farmacêutico   84.0    100.000000   0.000000
 GGBR4           Siderurgia   98.0    100.000000   0.000000
 FLRY3

In [7]:
fig = px.bar(
    company_stats.sort_values('match_rate_%'),
    x='match_rate_%', y='ticker',
    color='sector',
    title='Match Rate by Company (sorted ascending)',
    labels={'match_rate_%': 'Match Rate (%)', 'ticker': 'Ticker'},
    orientation='h',
    height=900,
)
fig.add_vline(x=90, line_dash='dash', line_color='red', annotation_text='90% target')
fig.show()


## 4. Failure Taxonomy

Classify why metrics are missing or wrong.

In [8]:
# Missing breakdown by metric (EBITDA + net_debt expected to be missing — no CVM line item)
missing = metrics[metrics.match_status == 'missing']
print("Missing metrics (top 15 by count):")
print(missing.metric_name.value_counts().head(15).to_string())

print()
print("Mismatch cases (top 15):")
mismatches = metrics[metrics.match_status == 'mismatch'].copy()
mismatches['pct_err_%'] = mismatches['percentage_error'] * 100
print(
    mismatches[['filing_id','metric_name','extracted_value','validated_value','pct_err_%']]
    .sort_values('pct_err_%', ascending=False)
    .head(15)
    .to_string(index=False)
)


Missing metrics (top 15 by count):
metric_name
cogs                   32
operating_cash_flow    17
gross_profit            7
revenue                 7
total_assets            4
net_income              2
total_equity            1

Mismatch cases (top 15):
             filing_id         metric_name  extracted_value  validated_value   pct_err_%
IRBR3_ITR_2023-06-30_1 operating_cash_flow        -849065.0         -28094.0 2922.228946
IRBR3_ITR_2023-06-30_2 operating_cash_flow        -849065.0         -28094.0 2922.228946
MULT3_ITR_2023-09-30_1          net_income       13849047.0         717930.0 1829.024696
IRBR3_ITR_2023-03-31_1 operating_cash_flow        -719274.0          41783.0 1821.451308
IRBR3_ITR_2023-09-30_2 operating_cash_flow        -966731.0         -57226.0 1589.321288
IRBR3_ITR_2023-09-30_1 operating_cash_flow        -966731.0         -57226.0 1589.321288
HAPV3_ITR_2024-03-31_1                cogs       -4916913.0        -327747.0 1400.216020
MULT3_ITR_2023-09-30_1        gro

## 5. Filing Type Comparison (ITR vs DFP)

In [9]:
filing_metrics = pd.read_sql("""
    SELECT f.filing_type, m.match_status, COUNT(*) as n
    FROM metrics m
    JOIN filings f ON m.filing_id = f.filing_id
    GROUP BY f.filing_type, m.match_status
""", conn)

pivot = filing_metrics.pivot(index='filing_type', columns='match_status', values='n').fillna(0)
pivot['total'] = pivot.sum(axis=1)
for col in ['exact', 'close', 'mismatch', 'missing']:
    if col in pivot.columns:
        pivot[f'{col}_%'] = pivot[col] / pivot['total'] * 100

print("Match breakdown by filing type:")
print(pivot.to_string())


Match breakdown by filing type:
match_status  close  exact  mismatch  missing  total    exact_%   close_%  mismatch_%  missing_%
filing_type                                                                                     
DFP               4   1178        21       26   1229  95.850285  0.325468    1.708706   2.115541
ITR              11   3247       120       44   3422  94.886032  0.321449    3.506721   1.285798


## 6. SQLite DB Sanity Check

In [10]:
# Confirm no orphaned metrics or filings
orphan_metrics = conn.execute("""
    SELECT COUNT(*) FROM metrics m
    WHERE NOT EXISTS (SELECT 1 FROM filings f WHERE f.filing_id = m.filing_id)
""").fetchone()[0]

orphan_filings = conn.execute("""
    SELECT COUNT(*) FROM filings f
    WHERE NOT EXISTS (SELECT 1 FROM companies c WHERE c.ticker = f.ticker)
""").fetchone()[0]

print(f"Orphaned metrics (no parent filing): {orphan_metrics}")
print(f"Orphaned filings (no parent company): {orphan_filings}")

# Sample from the metrics table
print()
print("Sample metric records:")
sample = pd.read_sql("""
    SELECT m.filing_id, m.metric_name, m.extracted_value, m.validated_value, m.match_status
    FROM metrics m
    JOIN filings f ON m.filing_id = f.filing_id
    WHERE f.filing_type = 'DFP' AND m.match_status = 'exact'
    ORDER BY RANDOM()
    LIMIT 10
""", conn)
print(sample.to_string(index=False))


Orphaned metrics (no parent filing): 0
Orphaned filings (no parent company): 0

Sample metric records:
             filing_id         metric_name  extracted_value  validated_value match_status
CPLE6_DFP_2022-12-31_1        total_assets       49703700.0       49703700.0        exact
CSAN3_DFP_2023-12-31_2        gross_profit       10918601.0       10918601.0        exact
FLRY3_DFP_2024-12-31_1        total_equity        5374831.0        5374831.0        exact
VIVT3_DFP_2022-12-31_1        total_assets      119121483.0      119121483.0        exact
EGIE3_DFP_2022-12-31_1          net_income        2664568.0        2664568.0        exact
PETR4_DFP_2023-12-31_2        gross_profit      269933000.0      269933000.0        exact
LCAM3_DFP_2022-12-31_1 operating_cash_flow       -2608489.0       -2608489.0        exact
MULT3_DFP_2024-12-31_1        gross_profit        2121411.0        2121411.0        exact
RECV3_DFP_2022-12-31_1          net_income        1153391.0        1153391.0        exa